# AE4 · ETL Steam + Twitch

**Caso:** una desarrolladora o publicadora quiere saber si la audiencia que un juego tiene en Twitch durante su lanzamiento se asocia con el alcance que logra en Steam (owners estimados).

Este notebook hace el ETL completo con pandas. Cada etapa sigue el mismo orden: **evidencia** (qué muestran los datos), **decisión** (qué hacemos y por qué), **acción** (código corto) y **verificación** (comprobar que quedó como esperábamos).

| Sección | Consigna 3.3 |
|---|---|
| 0. Configuración | — |
| 1. Ingesta de cada fuente por separado | 3.3.1 |
| 2. Perfilado inicial (nulos, duplicados, tipos, atípicos) | 3.3.2 |
| 3. Limpieza y transformación de Steam | 3.3.2 y 3.3.3 |
| 4. Limpieza y transformación de Twitch | 3.3.2 y 3.3.3 |
| 5. Merge entre fuentes | 3.3.4 |
| 6. Validaciones posteriores al merge | 3.3.5 |
| 7. Exportación para el Data Warehouse | — |

Para reproducirlo hay que ejecutar las celdas en orden con `games.csv` y `Twitch_game_data.csv` en la misma carpeta. Las salidas quedan en `salida_etl/`, y el reporte de la corrida, en `salida_etl/validaciones.txt`.

## 0. Configuración

Rutas, ventana de lanzamientos de la cohorte (enero de 2016 a junio de 2024) y dos funciones auxiliares: `log` guarda cada resultado para el reporte final y `registrar_paso` arma la tabla de trazabilidad de filas (cuántas filas entran y salen en cada paso).

In [1]:
from pathlib import Path
import re
import unicodedata

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

import sys
print(f"Python {sys.version.split()[0]} | pandas {pd.__version__}")   # versiones usadas, para reproducir

BASE_DIR = Path.cwd()
if not (BASE_DIR / "games.csv").exists() and (BASE_DIR.parent / "games.csv").exists():
    BASE_DIR = BASE_DIR.parent

RUTA_STEAM = BASE_DIR / "games.csv"
RUTA_TWITCH = BASE_DIR / "Twitch_game_data.csv"
SALIDA = BASE_DIR / "salida_etl"
SALIDA.mkdir(parents=True, exist_ok=True)

for ruta in (RUTA_STEAM, RUTA_TWITCH):
    if not ruta.exists():
        raise FileNotFoundError(f"No se encontró el archivo requerido: {ruta}")

VENTANA_DESDE = pd.Timestamp("2016-01-01")
VENTANA_HASTA = pd.Timestamp("2024-06-30")

REPORTE = []      # líneas de validaciones.txt
TRAZA = []        # trazabilidad de filas por paso


def log(msg=""):
    print(msg)
    REPORTE.append(str(msg))


def registrar_paso(fuente, paso, antes, despues):
    TRAZA.append({"fuente": fuente, "paso": paso, "filas_antes": antes,
                  "filas_despues": despues, "eliminadas": antes - despues})
    log(f"[{fuente}] {paso}: {antes:,} -> {despues:,} (eliminadas: {antes - despues:,})")

Python 3.12.5 | pandas 2.3.0


### Funciones de texto

Twitch y Steam **no comparten un identificador**, así que el cruce se hace por nombre. Para eso se usan dos funciones:

- `arreglar_texto`: repara texto que quedó mal decodificado (por ejemplo `PokÃ©mon` → `Pokémon`) y los códigos `<U+XXXX>`. Si la reparación falla, devuelve el texto original sin cambios.
- `normalizar`: genera la **clave de cruce**. Quita los símbolos ™ ® ©, los acentos, las mayúsculas y todo lo que no sea letra o número. Así `"Counter-Strike: Global Offensive"` pasa a `"counterstrikeglobaloffensive"`.

In [2]:
def arreglar_texto(texto):
    if not isinstance(texto, str):
        return texto
    texto = re.sub(r"<U\+([0-9A-Fa-f]{4})>", lambda m: chr(int(m.group(1), 16)), texto)
    try:
        return texto.encode("cp1252").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return texto


def normalizar(nombre):
    nombre = re.sub(r"[™®©]", "", str(nombre))
    nombre = unicodedata.normalize("NFKD", nombre).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]", "", nombre.lower())


for ejemplo in ["Counter-Strike: Global Offensive", "HELLDIVERS™ 2", "Pokémon Scarlet", "黑神话：悟空"]:
    print(f"{ejemplo!r:40} -> {normalizar(ejemplo)!r}")

'Counter-Strike: Global Offensive'       -> 'counterstrikeglobaloffensive'
'HELLDIVERS™ 2'                          -> 'helldivers2'
'Pokémon Scarlet'                        -> 'pokemonscarlet'
'黑神话：悟空'                                 -> ''


El último ejemplo es importante: **un nombre escrito solo con caracteres no latinos queda con clave vacía (`''`)**. Si no se tratan aparte, todos esos juegos comparten la misma clave y la deduplicación los junta en uno solo. Esto se resuelve en el paso 3.9.

## 1. Ingesta de cada fuente por separado

### 1.1 Steam (`games.csv`)

**Evidencia:** la primera línea del archivo tiene 39 nombres, pero cada fila tiene 40 valores. El encabezado trae `DiscountDLC count` pegado, cuando en realidad son dos columnas (`Discount` y `DLC count`).

In [3]:
with open(RUTA_STEAM, encoding="utf-8-sig") as f:
    encabezado = f.readline().strip().split(",")
print("Columnas declaradas en el encabezado:", len(encabezado))
print([c for c in encabezado if "Discount" in c])

Columnas declaradas en el encabezado: 39
['DiscountDLC count']


**Decisión:** se lee el archivo descartando ese encabezado (`header=0`) y se asignan a mano los 40 nombres correctos. Si no se hiciera, pandas correría todas las columnas un lugar y, por ejemplo, `Price` terminaría conteniendo el descuento.

In [4]:
COLS_STEAM = [
    "AppID", "Name", "Release date", "Estimated owners", "Peak CCU", "Required age",
    "Price", "Discount", "DLC count", "About the game", "Supported languages",
    "Full audio languages", "Reviews", "Header image", "Website", "Support url",
    "Support email", "Windows", "Mac", "Linux", "Metacritic score", "Metacritic url",
    "User score", "Positive", "Negative", "Score rank", "Achievements",
    "Recommendations", "Notes", "Average playtime forever",
    "Average playtime two weeks", "Median playtime forever",
    "Median playtime two weeks", "Developers", "Publishers", "Categories",
    "Genres", "Tags", "Screenshots", "Movies",
]

log("=" * 70 + "\n1. INGESTA\n" + "=" * 70)
steam_raw = pd.read_csv(RUTA_STEAM, encoding="utf-8-sig", header=0, names=COLS_STEAM, low_memory=False)
assert steam_raw.shape[1] == 40, "Steam no tiene las 40 columnas esperadas"
log(f"Steam: {steam_raw.shape[0]:,} filas x {steam_raw.shape[1]} columnas")
steam_raw[["AppID", "Name", "Release date", "Estimated owners", "Price", "Positive", "Negative", "Genres"]].head()

1. INGESTA


Steam: 125,855 filas x 40 columnas


,AppID,Name,Release date,Estimated owners,Price,Positive,Negative,Genres
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0.00,0,0,NaN
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,5.24,252,3,Adventure
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,4.99,21,3,Casual
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0 - 20000,8.99,0,0,"Casual,Indie,Simulation"
4,3631080,Maze Quest VR,"Apr 24, 2025",0 - 20000,4.99,0,0,"Action,Early Access"


### 1.2 Twitch (`Twitch_game_data.csv`)

**Evidencia:** si se lee como UTF-8, el archivo da error en algunos bytes. Esos bytes corresponden a la codificación Windows-1252.

**Decisión:** se lee con `encoding="cp1252"`. Los nombres que de todas formas quedan mal decodificados se reparan después con `arreglar_texto` (paso 4.3).

In [5]:
try:
    pd.read_csv(RUTA_TWITCH, encoding="utf-8")
    print("UTF-8: se pudo leer")
except UnicodeDecodeError as e:
    print("UTF-8 falla:", str(e)[:90])

twitch_raw = pd.read_csv(RUTA_TWITCH, encoding="cp1252")
log(f"Twitch: {twitch_raw.shape[0]:,} filas x {twitch_raw.shape[1]} columnas")
twitch_raw.head()

UTF-8 falla: 'utf-8' codec can't decode byte 0x96 in position 30: invalid start byte
Twitch: 21,000 filas x 12 columnas


,Rank,Game,Month,Year,Hours_watched,Hours_streamed,Peak_viewers,Peak_channels,Streamers,Avg_viewers,Avg_channels,Avg_viewer_ratio
0,1,League of Legends,1,2016,94377226,1362044,530270,2903,129172,127021,1833,69.29
1,2,Counter-Strike: Global Offensive,1,2016,47832863,830105,372654,2197,120849,64378,1117,57.62
2,3,Dota 2,1,2016,45185893,433397,315083,1100,44074,60815,583,104.26
3,4,Hearthstone,1,2016,39936159,235903,131357,517,36170,53749,317,169.29
4,5,Call of Duty: Black Ops III,1,2016,16153057,1151578,71639,3620,214054,21740,1549,14.03


**Columnas que se usan:** de las 40 columnas de Steam alcanzan con nombre, fecha, owners, precio, reseñas, género, desarrollador y publicador (las que aparecen en el contexto del caso). Las demás (capturas, URLs, descripciones) no responden ninguna pregunta de negocio y se descartan al final de la limpieza. De Twitch se conservan las métricas mensuales.

## 2. Perfilado inicial

Antes de limpiar se mide el estado de los datos. Cada decisión de limpieza de las secciones 3 y 4 se apoya en lo que se ve acá.

### 2.1 Tipos de datos

In [6]:
COLS_USADAS_STEAM = ["AppID", "Name", "Release date", "Estimated owners", "Peak CCU", "Price",
                     "Positive", "Negative", "Recommendations", "Genres", "Developers", "Publishers"]
pd.DataFrame({
    "tipo": steam_raw[COLS_USADAS_STEAM].dtypes.astype(str),
    "ejemplo": steam_raw[COLS_USADAS_STEAM].iloc[1],
})

,tipo,ejemplo
AppID,int64,496350
Name,object,Supipara - Chapter 1 Spring Has Come!
Release date,object,"Jul 29, 2016"
Estimated owners,object,0 - 20000
Peak CCU,int64,0
Price,float64,5.24
Positive,int64,252
Negative,int64,3
Recommendations,int64,231
Genres,object,Adventure


`Release date` es texto (`"Aug 1, 2023"`) y `Estimated owners` es un **rango** en texto (`"0 - 20000"`). Las dos columnas deben convertirse para poder filtrar por fecha y calcular el KPI 2. En Twitch, año y mes vienen en columnas separadas:

In [7]:
twitch_raw.dtypes.to_frame("tipo").T

,Rank,Game,Month,Year,Hours_watched,Hours_streamed,Peak_viewers,Peak_channels,Streamers,Avg_viewers,Avg_channels,Avg_viewer_ratio
tipo,int64,object,int64,int64,int64,int64,int64,int64,int64,int64,int64,float64


### 2.2 Nulos

In [8]:
def resumen_nulos(df, columnas):
    n = df[columnas].isna().sum()
    return pd.DataFrame({"nulos": n, "%": (n / len(df) * 100).round(2)}).query("nulos > 0")

print("Steam (columnas usadas):")
display(resumen_nulos(steam_raw, COLS_USADAS_STEAM))
print("Twitch (todas las columnas):")
display(resumen_nulos(twitch_raw, twitch_raw.columns.tolist()))

Steam (columnas usadas):


,nulos,%
Name,1,0.00
Genres,8423,6.69
Developers,8449,6.71
Publishers,8922,7.09


Twitch (todas las columnas):


,nulos,%
Game,1,0.0


**Lectura:**
- `Name` (Steam) y `Game` (Twitch) tienen 1 nulo cada uno. Sin nombre no hay clave de cruce, así que esas filas se eliminan (pasos 3.1 y 4.1).
- `Genres`, `Developers` y `Publishers` tienen alrededor de un 7 % de nulos (la mayoría corresponde a productos `0 - 0`, que se excluyen en el paso 3.4). **No se eliminan** esas filas porque el juego sigue sirviendo para P1, P3 y P4. El género faltante se reemplaza por la categoría explícita `"Sin género"` para que P2 lo muestre aparte en vez de perderlo.
- Las métricas numéricas de las dos fuentes no tienen nulos.

### 2.3 Duplicados

In [9]:
print("Steam - filas idénticas:", steam_raw.duplicated().sum())
print("Steam - AppID repetidos:", steam_raw["AppID"].duplicated().sum())
print("Twitch - filas idénticas:", twitch_raw.duplicated().sum())
print("Twitch - (Game, Year, Month) repetidos:", twitch_raw.duplicated(["Game", "Year", "Month"]).sum())

Steam - filas idénticas: 0
Steam - AppID repetidos: 0
Twitch - filas idénticas: 0
Twitch - (Game, Year, Month) repetidos: 10


No hay filas idénticas ni AppID repetidos. En Twitch hay **10 pares (juego, mes) repetidos** con valores distintos: Twitch tuvo dos categorías con el mismo nombre en ese mes (por ejemplo *Among Us* en enero de 2023). Además, el cruce se hace por el **nombre normalizado**, así que aparecen más duplicados después de normalizar y aplicar alias. Todo esto se trata en los pasos 3.9 (Steam) y 4.6 (Twitch).

### 2.4 Rangos y valores atípicos

In [10]:
display(steam_raw[["Price", "Peak CCU", "Positive", "Negative"]].describe().T)
display(twitch_raw[["Hours_watched", "Peak_viewers", "Streamers", "Avg_viewers"]].describe().T)

,count,mean,std,min,25%,50%,75%,max
Price,125855.0,4.811174,12.489329,0.0,0.59,2.39,5.545,999.98
Peak CCU,125855.0,53.187200,3681.082965,0.0,0.00,0.00,0.000,1013936.00
Positive,125855.0,1018.182091,27727.824406,0.0,0.00,4.00,35.000,7642084.00
Negative,125855.0,164.878630,5305.000665,0.0,0.00,1.00,10.000,1173003.00


,count,mean,std,min,25%,50%,75%,max
Hours_watched,21000.0,5.831853e+06,1.943051e+07,89811.0,543089.75,1079297.0,3032041.00,344551979.0
Peak_viewers,21000.0,6.430924e+04,1.554388e+05,441.0,10048.25,23459.5,55190.00,3915488.0
Streamers,21000.0,2.365554e+04,7.051357e+04,0.0,1916.00,5374.5,14389.75,1728322.0
Avg_viewers,21000.0,7.996369e+03,2.664764e+04,120.0,744.00,1481.0,4135.50,479209.0


In [11]:
steam_raw["Estimated owners"].value_counts().to_frame("juegos")

,juegos
Estimated owners,
0 - 20000,75705
0 - 0,24579
20000 - 50000,11397
50000 - 100000,5355
100000 - 200000,3454
200000 - 500000,2856
500000 - 1000000,1155
1000000 - 2000000,729
2000000 - 5000000,405


**Lectura:**
- Ningún precio es negativo. El máximo ronda los 1.000 USD, muy lejos de la mediana, así que los precios mayores a 100 USD se analizan en el paso 3.7.
- Owners, reseñas y horas vistas tienen **colas largas**: pocos juegos concentran casi todo. Ese es el fenómeno que se estudia, no un error de carga (paso 3.7).
- Alrededor del 80 % del catálogo está en `0 - 0` o `0 - 20000`. El rango `0 - 0` se trata en el paso 3.4.
- En Twitch, `Streamers` llega a valer 0 en algunas filas (se analiza en el paso 4.2).

## 3. Limpieza y transformación de Steam

Se trabaja sobre una copia (`steam`) para no modificar los datos crudos. Cada paso registra cuántas filas entran y cuántas salen.

### 3.1 Nulos en el nombre

In [12]:
log("\n" + "=" * 70 + "\n3. LIMPIEZA STEAM\n" + "=" * 70)
steam = steam_raw.copy()
display(steam.loc[steam["Name"].isna(), ["AppID", "Name", "Release date", "Estimated owners"]])

antes = len(steam)
steam = steam.dropna(subset=["Name"])
registrar_paso("Steam", "Eliminar filas sin nombre", antes, len(steam))


3. LIMPIEZA STEAM


,AppID,Name,Release date,Estimated owners
44432,396420,NaN,"Nov 1, 2016",0 - 0


[Steam] Eliminar filas sin nombre: 125,855 -> 125,854 (eliminadas: 1)


### 3.2 Duplicados por `AppID`

`AppID` identifica un producto en Steam. Ya se vio que no hay repetidos, pero la operación se deja explícita (con `keep="first"`) para que el ETL siga siendo correcto si se vuelve a correr con otra descarga del dataset.

In [13]:
antes = len(steam)
steam = steam.drop_duplicates("AppID", keep="first")
registrar_paso("Steam", "Eliminar AppID duplicados", antes, len(steam))

[Steam] Eliminar AppID duplicados: 125,854 -> 125,854 (eliminadas: 0)


### 3.3 Tipos de datos y formatos

- `AppID` → entero.
- `Price` → número decimal. **Ya está en USD con decimales** (`19.99`), así que no se divide por 100.
- `Release date` → fecha con formato `"Mes Día, Año"` (`"Aug 1, 2023"`). Antes de convertir se verifica con una expresión regular que **todas** las filas sigan ese formato. Con `errors="coerce"`, una fecha que no se pueda interpretar queda como `NaT` en lugar de cortar el proceso, y **se cuenta** para que ninguna fila quede afuera sin que se informe.

In [14]:
steam["AppID"] = steam["AppID"].astype("int64")
steam["precio_usd"] = pd.to_numeric(steam["Price"], errors="coerce")
steam["fecha_lanzamiento"] = pd.to_datetime(steam["Release date"], format="mixed", errors="coerce")

log(f"Fechas no interpretables (NaT): {steam['fecha_lanzamiento'].isna().sum()}")
log(f"Precios no numéricos (NaN): {steam['precio_usd'].isna().sum()}")
print("Rango de fechas:", steam["fecha_lanzamiento"].min().date(), "a", steam["fecha_lanzamiento"].max().date())

formato_ok = steam["Release date"].str.fullmatch(r"[A-Z][a-z]{2} \d{1,2}, \d{4}")
log(f"Fechas con formato 'Mes Día, Año': {formato_ok.sum():,} de {len(steam):,}")
if (~formato_ok).any():
    display(steam.loc[~formato_ok, "Release date"].value_counts().head())

Fechas no interpretables (NaT): 0
Precios no numéricos (NaN): 0
Rango de fechas: 1997-06-30 a 2026-12-01
Fechas con formato 'Mes Día, Año': 125,854 de 125,854


Todas las fechas tienen el mismo formato, y fechas y precios se convirtieron sin generar nulos. El dataset incluye lanzamientos posteriores a junio de 2024 (incluso fechas anunciadas a futuro). No se borran porque siguen contando como catálogo, pero quedan **fuera de la cohorte** de análisis a través de la marca `en_ventana` (paso 3.10).

### 3.4 Owners: del rango al punto medio (KPI 2)

**Evidencia:** `Estimated owners` es un rango publicado (`"20000 - 50000"`) y no un número exacto.

**Decisión:**
- Se separa en `owners_min` y `owners_max`, y se calcula `owners_medio = (min + max) / 2`, que es la fórmula del KPI 2.
- **Se excluye `0 - 0`.** Ese rango no indica "cero jugadores": corresponde a productos que todavía no salieron, se retiraron o no tienen estimación. Mantenerlos pondría un 0 falso en el promedio.
- **Se conserva `0 - 20000`** (punto medio 10.000) como señal baja. Es un rango válido, y sacarlo eliminaría a la mayoría de los juegos chicos y sesgaría la muestra. Se marca con `owners_baja_senal` para poder filtrarlo en el dashboard.

In [15]:
rango = steam["Estimated owners"].str.split(" - ", expand=True).astype(float)
steam["owners_min"], steam["owners_max"] = rango[0], rango[1]
steam["owners_medio"] = (steam["owners_min"] + steam["owners_max"]) / 2
steam["owners_baja_senal"] = steam["owners_max"] <= 20000

assert (steam["owners_min"] <= steam["owners_max"]).all(), "Hay rangos invertidos"

antes = len(steam)
steam = steam[steam["owners_max"] > 0].copy()
registrar_paso("Steam", "Excluir owners '0 - 0'", antes, len(steam))

(steam.groupby(["Estimated owners", "owners_medio"]).size().rename("juegos")
      .reset_index().sort_values("owners_medio"))

[Steam] Excluir owners '0 - 0': 125,854 -> 101,276 (eliminadas: 24,578)


,Estimated owners,owners_medio,juegos
0,0 - 20000,10000.0,75705
5,20000 - 50000,35000.0,11397
9,50000 - 100000,75000.0,5355
1,100000 - 200000,150000.0,3454
6,200000 - 500000,350000.0,2856
10,500000 - 1000000,750000.0,1155
2,1000000 - 2000000,1500000.0,729
7,2000000 - 5000000,3500000.0,405
11,5000000 - 10000000,7500000.0,125
3,10000000 - 20000000,15000000.0,51


### 3.5 Productos que no son el juego base

**Evidencia:** el catálogo de Steam incluye demos, playtests, bandas sonoras y servidores dedicados. Esos productos comparten nombre con el juego y pueden "robarle" el cruce con Twitch o duplicar conteos.

**Decisión:** se eliminan los títulos que contienen esas palabras completas (se usa `\b` para no borrar, por ejemplo, un juego llamado *Demolition*).

In [16]:
REGEX_NO_BASE = re.compile(r"\b(?:playtest|demo|soundtrack|dedicated server)\b", re.I)

steam["nombre"] = steam["Name"].map(arreglar_texto)
es_no_base = steam["nombre"].str.contains(REGEX_NO_BASE, na=False)
display(steam.loc[es_no_base, ["AppID", "nombre"]].sample(8, random_state=1))

antes = len(steam)
steam = steam[~es_no_base].copy()
registrar_paso("Steam", "Excluir demos/playtests/soundtracks/servidores", antes, len(steam))

,AppID,nombre
70386,2824760,Brass Necessity - Demo
83553,1434340,Insect Adventure Demo
92848,1551110,Mythic Prelude - Concept Demo
119333,831230,Doors Quest Demo
5289,1102770,Cappasity VR Store Demo
7553,2934870,Cyberpunk Sister Playtest
104968,1792070,The Faded - Chapter 1 - The Perish Forest Demo
72203,2169160,Hero Jumper Demo


[Steam] Excluir demos/playtests/soundtracks/servidores: 101,276 -> 101,227 (eliminadas: 49)


### 3.6 Reparación de texto

Se cuenta cuántos nombres modificó `arreglar_texto`. Steam se leyó como UTF-8 correcto, así que se espera que no modifique ninguno: eso confirma que la función no altera nombres que ya estaban bien. En Twitch sí repara nombres (paso 4.3).

In [17]:
cambiados = steam["nombre"] != steam["Name"]
log(f"Nombres de Steam reparados por codificación: {cambiados.sum():,}")
steam.loc[cambiados, ["Name", "nombre"]].head(5)

Nombres de Steam reparados por codificación: 0


,Name,nombre


### 3.7 Valores atípicos

**Precio:** los productos de más de 100 USD son pocos (menos del 0,2 %) y muchos son software, experiencias VR comerciales o ediciones especiales.

In [18]:
steam["precio_atipico"] = steam["precio_usd"] > 100
log(f"Juegos con precio > 100 USD (marcados, no eliminados): {steam['precio_atipico'].sum():,}")
steam.loc[steam["precio_atipico"], ["nombre", "precio_usd", "owners_medio"]].sort_values("precio_usd", ascending=False).head(6)

Juegos con precio > 100 USD (marcados, no eliminados): 188


,nombre,precio_usd,owners_medio
41088,The Leverage Game Business Edition,999.98,10000.0
68275,The Leverage Game,999.98,10000.0
34463,Ascent Free-Roaming VR Experience,999.00,10000.0
31935,True Love,500.00,10000.0
77652,Aartform Curvy 3D 4.0,299.99,10000.0
85479,Aartform Curvy 3D 3.0,299.90,10000.0


**Decisión sobre el precio:** se **marcan** con `precio_atipico` y no se eliminan. Son precios reales, no errores de carga, y el precio no se usa en ningún KPI. Desde el dashboard se pueden excluir con un filtro.

**Owners y reseñas:** se mide cuántos juegos superan el límite superior del criterio IQR (Q3 + 1,5·IQR).

In [19]:
def contar_atipicos_iqr(serie):
    q1, q3 = serie.quantile([0.25, 0.75])
    limite = q3 + 1.5 * (q3 - q1)
    return pd.Series({"Q1": q1, "Q3": q3, "límite IQR": limite,
                      "atípicos": int((serie > limite).sum()), "% atípicos": round((serie > limite).mean() * 100, 1)})

steam["resenas_total"] = steam["Positive"] + steam["Negative"]
pd.DataFrame({c: contar_atipicos_iqr(steam[c]) for c in ["owners_medio", "resenas_total"]}).T

,Q1,Q3,límite IQR,atípicos,% atípicos
owners_medio,10000.0,35000.0,72500.0,14168.0,14.0
resenas_total,2.0,79.0,194.5,16815.0,16.6


**Decisión sobre owners y reseñas:** **no se eliminan**. En un mercado de "pocos éxitos y muchos juegos chicos", lo que el criterio IQR detecta como atípico son justamente los juegos exitosos (CS2, Palworld, Helldivers 2). Borrarlos dejaría sin sentido las preguntas P1 y P3. Para que la cola larga no distorsione los resultados, el análisis usa **cuartiles y medianas** en lugar de promedios.

### 3.8 Género principal (P2)

`Genres` es una lista separada por comas. La forma más simple sería tomar el primero, pero antes hay que ver **en qué orden vienen**:

In [20]:
listas_generos = steam["Genres"].dropna().str.split(",")
listas_tags = steam["Tags"].dropna().str.split(",")
print(f"Listas de Genres en orden alfabético: {listas_generos.map(lambda g: g == sorted(g)).mean():.1%}")
print(f"Listas de Tags en orden alfabético:   {listas_tags.map(lambda t: t == sorted(t)).mean():.1%}")
steam.loc[steam["nombre"].isin(["Euro Truck Simulator 2", "Unrailed!", "Trine 4: The Nightmare Prince"]),
          ["nombre", "Genres", "Tags"]].assign(Tags=lambda d: d["Tags"].str[:70])

Listas de Genres en orden alfabético: 87.3%
Listas de Tags en orden alfabético:   4.0%


,nombre,Genres,Tags
732,Trine 4: The Nightmare Prince,"Action,Adventure,Indie","Puzzle-Platformer,Local Co-Op,Online Co-Op,Act..."
1111,Euro Truck Simulator 2,"Indie,Simulation","Driving,Transportation,Simulation,Open World,A..."
1566,Unrailed!,"Action,Casual,Indie","Multiplayer,Co-op,Online Co-Op,Local Co-Op,Cas..."


**Evidencia:** casi 9 de cada 10 listas de `Genres` vienen **en orden alfabético**. Entonces "el primer género" es casi siempre *Action*, *Adventure* o *Casual* por orden de abecedario, no por ser el principal. *Euro Truck Simulator 2* (`Indie,Simulation`) quedaría como *Indie*, y RPG o Strategy, que P2 nombra explícitamente, quedarían subrepresentados.

En cambio, `Tags` **no** está en orden alfabético: Steam ordena las etiquetas según cuántos usuarios las votaron, así que la primera es la que más identifica al juego.

**Decisión:** `genero_principal` = el **primer tag (el más votado) que además figure en `Genres`**. Así se respeta la lista oficial de géneros, pero se elige el más representativo. `Early Access` y `Free To Play` no se toman como género porque describen un modelo de venta, no un tipo de juego. Si ningún tag coincide, se usa el primer género válido, y si no hay ninguno, `"Sin género"` (ver la decisión del paso 2.2).

In [21]:
NO_SON_GENERO = {"Early Access", "Free To Play"}


def elegir_genero(generos, tags):
    if not isinstance(generos, str):
        return "Sin género"
    validos = [g.strip() for g in generos.split(",") if g.strip() not in NO_SON_GENERO]
    if not validos:
        return "Sin género"
    for tag in (tags.split(",") if isinstance(tags, str) else []):
        if tag.strip() in validos:
            return tag.strip()
    return validos[0]


steam["genero_principal"] = [elegir_genero(g, t) for g, t in zip(steam["Genres"], steam["Tags"])]
genero_alfabetico = steam["Genres"].fillna("").str.split(",").str[0].replace("", "Sin género")

log(f"Juegos sin género informado -> 'Sin género': {(steam['genero_principal'] == 'Sin género').sum():,}")
log(f"Juegos cuyo género principal cambia respecto de 'primer género alfabético': "
    f"{(steam['genero_principal'] != genero_alfabetico).sum():,}")
pd.DataFrame({"primer_genero_alfabetico": genero_alfabetico.value_counts(),
              "genero_principal (por tags)": steam["genero_principal"].value_counts()}
             ).fillna(0).astype(int).sort_values("genero_principal (por tags)", ascending=False).head(12)

Juegos sin género informado -> 'Sin género': 227
Juegos cuyo género principal cambia respecto de 'primer género alfabético': 34,045


,primer_genero_alfabetico,genero_principal (por tags)
Action,41082,27419
Casual,20379,19663
Adventure,21488,17009
Indie,9659,12367
Strategy,1438,7596
RPG,1707,6620
Simulation,2203,5876
Racing,501,1519
Sports,195,1134
Utilities,310,588


### 3.9 Clave de cruce y nombres repetidos

Se genera `clave = normalizar(nombre)`. Hay dos problemas por resolver.

**a) Claves vacías.** Los nombres escritos solo en chino, japonés, coreano, etc. quedan con clave `''`.

In [22]:
steam["clave"] = steam["nombre"].map(normalizar)
vacias = steam["clave"] == ""
log(f"Juegos con nombre no latino (clave vacía): {vacias.sum():,}")
display(steam.loc[vacias, ["AppID", "nombre", "owners_medio"]].head(5))

steam.loc[vacias, "clave"] = "appid" + steam.loc[vacias, "AppID"].astype(str)
steam["clave_cruzable"] = ~vacias

Juegos con nombre no latino (clave vacía): 1,470


,AppID,nombre,owners_medio
55,1679730,筠宝大冒险,10000.0
129,2823420,欧派大作战,10000.0
407,2558480,碧海潮生怀旧版,10000.0
433,3434240,《'点'到为止》,10000.0
461,3432210,吃脚,10000.0


**Decisión:** a esos juegos se les asigna una clave única basada en el AppID (`appid12345`) y se marcan con `clave_cruzable = False`. Así **siguen en el universo de Steam** (cuentan en P3 y en el KPI 3 como "sin presencia observada"), pero no pueden cruzar con Twitch por error.

> En la versión anterior del ETL todos compartían la clave `''` y la deduplicación los reducía a uno solo, de modo que **se perdían alrededor de 1.470 juegos** sin que se informara.

**b) Varios AppID con el mismo nombre.** Por ejemplo, ediciones viejas y nuevas con el mismo título. Si la clave no es única en Steam, el merge multiplicaría las filas de Twitch.

In [23]:
steam["n_appid_mismo_nombre"] = steam.groupby("clave")["AppID"].transform("size")
conflictos = steam[steam["n_appid_mismo_nombre"] > 1]
print(f"Claves repetidas: {conflictos['clave'].nunique():,} ({len(conflictos):,} filas)")
(conflictos.sort_values(["clave", "owners_medio"], ascending=[True, False])
           .loc[conflictos["clave"].isin(conflictos.nlargest(4, "owners_medio")["clave"]),
                ["clave", "AppID", "nombre", "fecha_lanzamiento", "owners_medio", "resenas_total"]])

Claves repetidas: 1,269 (2,900 filas)


,clave,AppID,nombre,fecha_lanzamiento,owners_medio,resenas_total
91250,counterstrikeconditionzero,100,Counter-Strike: Condition Zero,2004-03-01,15000000.0,14977
89450,counterstrikeconditionzero,80,Counter-Strike: Condition Zero,2004-03-01,7500000.0,26558
104923,portal,400,Portal,2007-10-10,15000000.0,177741
101980,portal,52003,Portal,2007-10-10,10000.0,57774
10714,sidmeierscivilizationv,8930,Sid Meier's Civilization® V,2010-09-21,15000000.0,204265
105917,sidmeierscivilizationv,50100,Sid Meier's Civilization® V,2010-09-21,10000.0,140437
40969,tomclancysrainbowsixsiegex,359550,Tom Clancy's Rainbow Six® Siege X,2015-12-01,35000000.0,1398584
9187,tomclancysrainbowsixsiegex,488821,Tom Clancy's Rainbow Six® Siege X,2015-12-01,10000.0,376369
10471,tomclancysrainbowsixsiegex,488822,Tom Clancy's Rainbow Six® Siege X,2015-12-01,10000.0,376548
38815,tomclancysrainbowsixsiegex,488823,Tom Clancy's Rainbow Six® Siege X,2015-12-01,10000.0,376907


**Decisión:** por cada clave se conserva el AppID con **más owners**, y si hay empate, el de **más reseñas**. Es el producto principal y el que con más probabilidad corresponde a lo que se transmite en Twitch. La columna `n_appid_mismo_nombre` queda en la salida para poder auditar estos casos. Si en algún caso se elige una edición incorrecta, la marca `match_sospechoso` del paso 5.5 ayuda a detectarlo.

In [24]:
antes = len(steam)
steam = (steam.sort_values(["owners_medio", "resenas_total"], ascending=False)
              .drop_duplicates("clave", keep="first"))
registrar_paso("Steam", "Resolver nombres repetidos (1 AppID por clave)", antes, len(steam))
assert steam["clave"].is_unique

[Steam] Resolver nombres repetidos (1 AppID por clave): 101,227 -> 99,596 (eliminadas: 1,631)


### 3.10 Cohorte de análisis y columnas finales

`en_ventana` marca los lanzamientos entre enero de 2016 y junio de 2024. Se cortan ahí porque Twitch tiene datos hasta septiembre de 2024: un juego lanzado en junio de 2024 todavía tiene sus 3 meses completos de observación (junio, julio y agosto). Los juegos fuera de la ventana **no se borran**: siguen sirviendo para P3, que no depende de la fecha.

In [25]:
steam["en_ventana"] = steam["fecha_lanzamiento"].between(VENTANA_DESDE, VENTANA_HASTA)
log(f"Juegos dentro de la ventana 2016-01 / 2024-06: {steam['en_ventana'].sum():,}")

COLS_STEAM_FINAL = [
    "AppID", "nombre", "clave", "clave_cruzable", "fecha_lanzamiento", "en_ventana",
    "precio_usd", "precio_atipico", "owners_min", "owners_max", "owners_medio", "owners_baja_senal",
    "Peak CCU", "Positive", "Negative", "resenas_total", "Recommendations",
    "genero_principal", "Genres", "Developers", "Publishers", "n_appid_mismo_nombre",
]
steam_limpio = steam[COLS_STEAM_FINAL].reset_index(drop=True)
log(f"Steam limpio: {len(steam_raw):,} -> {len(steam_limpio):,} filas, {steam_limpio.shape[1]} columnas")
steam_limpio.dtypes.to_frame("tipo").T

Juegos dentro de la ventana 2016-01 / 2024-06: 70,708
Steam limpio: 125,855 -> 99,596 filas, 22 columnas


,AppID,nombre,clave,clave_cruzable,fecha_lanzamiento,en_ventana,precio_usd,precio_atipico,owners_min,owners_max,owners_medio,owners_baja_senal,Peak CCU,Positive,Negative,resenas_total,Recommendations,genero_principal,Genres,Developers,Publishers,n_appid_mismo_nombre
tipo,int64,object,object,bool,datetime64[ns],bool,float64,bool,float64,float64,float64,bool,int64,int64,int64,int64,int64,object,object,object,object,int64


## 4. Limpieza y transformación de Twitch

### 4.1 Nulos en el nombre del juego

In [26]:
log("\n" + "=" * 70 + "\n4. LIMPIEZA TWITCH\n" + "=" * 70)
twitch = twitch_raw.copy()
display(twitch[twitch["Game"].isna()])

antes = len(twitch)
twitch = twitch.dropna(subset=["Game"])
registrar_paso("Twitch", "Eliminar filas sin nombre de juego", antes, len(twitch))


4. LIMPIEZA TWITCH


,Rank,Game,Month,Year,Hours_watched,Hours_streamed,Peak_viewers,Peak_channels,Streamers,Avg_viewers,Avg_channels,Avg_viewer_ratio
7561,162,NaN,2,2019,317951,77004,30919,5804,21758,473,114,4.13


[Twitch] Eliminar filas sin nombre de juego: 21,000 -> 20,999 (eliminadas: 1)


### 4.2 Tipos, rangos y valores atípicos

Se comprueba que los meses estén entre 1 y 12, que las métricas no sean negativas y que cada mes tenga como mucho 200 juegos (el ranking es un top 200).

In [27]:
METRICAS_TW = ["Hours_watched", "Hours_streamed", "Peak_viewers", "Peak_channels",
               "Streamers", "Avg_viewers", "Avg_channels"]
assert twitch["Month"].between(1, 12).all(), "Hay meses fuera de rango"
assert (twitch[METRICAS_TW] >= 0).all().all(), "Hay métricas negativas"
assert twitch.groupby(["Year", "Month"]).size().max() <= 200, "Algún mes tiene más de 200 juegos"

twitch["fecha"] = pd.to_datetime(dict(year=twitch["Year"], month=twitch["Month"], day=1))
log(f"Período Twitch: {twitch['fecha'].min():%Y-%m} a {twitch['fecha'].max():%Y-%m} "
    f"({twitch['fecha'].nunique()} meses)")
log(f"Filas con Streamers = 0: {(twitch['Streamers'] == 0).sum()}")
twitch.loc[twitch["Streamers"] == 0, ["Game", "fecha", "Hours_watched", "Peak_viewers", "Streamers"]].head()

Período Twitch: 2016-01 a 2024-09 (105 meses)
Filas con Streamers = 0: 13


,Game,fecha,Hours_watched,Peak_viewers,Streamers
55,Star Wars Battlefront,2016-01-01,743255,7165,0
280,Star Wars Battlefront,2016-02-01,378280,5625,0
490,Star Wars Battlefront,2016-03-01,324185,11722,0
714,Star Wars Battlefront,2016-04-01,216470,35256,0
924,Star Wars Battlefront,2016-05-01,176777,2119,0


**Decisión:**
- Se arma `fecha` (primer día del mes) a partir de `Year` y `Month`, que es el formato que va a usar `DimTiempo`.
- Las filas con `Streamers = 0` **se conservan**. Tienen horas vistas y picos válidos, y el 0 refleja un límite de la fuente en ese mes, no un juego sin audiencia. Solo afectan a P4, donde se usa el **máximo** mensual, así que un 0 aislado no cambia el resultado.
- Las horas vistas también tienen cola larga (League of Legends o Just Chatting superan por mucho al resto). Es la realidad del ranking y **no se eliminan**, por el mismo criterio que en el paso 3.7.

### 4.3 Reparación de texto y clave

In [28]:
twitch["juego"] = twitch["Game"].map(arreglar_texto)
reparados = twitch["juego"] != twitch["Game"]
log(f"Nombres de Twitch reparados por codificación: {reparados.sum():,}")
display(twitch.loc[reparados, ["Game", "juego"]].drop_duplicates().head(5))

twitch["clave_original"] = twitch["juego"].map(normalizar)
assert (twitch["clave_original"] != "").all(), "Hay nombres de Twitch con clave vacía"

Nombres de Twitch reparados por codificación: 394


,Game,juego
50,Pok<U+00E9>mon Omega Ruby/Alpha Sapphire,Pokémon Omega Ruby/Alpha Sapphire
78,Pok<U+00E9>mon Red/Blue,Pokémon Red/Blue
79,Pok<U+00E9>mon Yellow,Pokémon Yellow
377,Pok<U+00E9>mon FireRed/LeafGreen,Pokémon FireRed/LeafGreen
450,Pokk<U+00E9>n Tournament,Pokkén Tournament


### 4.4 Categorías que no son videojuegos

**Evidencia:** el ranking de Twitch incluye categorías como *Just Chatting*, *Slots* o *Music*. No son juegos, no existen en Steam y algunas acumulan miles de millones de horas.

**Decisión:** se excluyen con una lista explícita y revisada a mano. Se usa una lista y no una regla automática para que cada exclusión se pueda justificar.

In [29]:
NO_JUEGOS = {
    "justchatting", "slots", "music", "specialevents", "sports", "irl",
    "talkshowspodcasts", "asmr", "art", "poker", "virtualcasino",
    "poolshottubsandbeaches", "traveloutdoors", "gamesdemos", "creative",
    "casinojackpot", "fooddrink", "casino", "retro", "imonlysleeping",
    "sciencetechnology", "politics", "fitnesshealth", "makerscrafting",
    "softwareandgamedevelopment", "animalsaquariumszoos", "gambling",
    "alwayson", "watchparties", "justsleeping", "coworkingstudying",
    "musicperformingarts", "chess", "dungeonsdragons",
    "casinoold", "casinoslotmachine", "crypto", "tabletoprpgs",
}

es_no_juego = twitch["clave_original"].isin(NO_JUEGOS)
display(twitch[es_no_juego].groupby("juego")["Hours_watched"].sum()
        .sort_values(ascending=False).head(10).to_frame("horas_vistas"))

antes = len(twitch)
twitch = twitch[~es_no_juego].copy()
registrar_paso("Twitch", "Excluir categorías que no son juegos", antes, len(twitch))

,horas_vistas
juego,
Just Chatting,14397469625
Slots,1208103879
Music,1147942190
Special Events,770536855
Sports,760527356
IRL,662772269
Talk Shows & Podcasts,566500310
ASMR,547766607
Art,537991909


[Twitch] Excluir categorías que no son juegos: 20,999 -> 19,109 (eliminadas: 1,890)


> A la lista original se le agregaron `Casino_old`, `Casino Slot Machine`, `Crypto` y `Tabletop RPGs`. Aparecían entre los "juegos sin cruce" con más horas y tampoco son videojuegos.

### 4.5 Alias manuales Twitch → Steam

**Evidencia:** el archivo `twitch_sin_match.csv` (diagnóstico generado en el paso 6) mostró juegos con muchas horas que no cruzaban porque **el mismo producto tiene otro nombre en Steam**. El caso más grave era *Fall Guys: Ultimate Knockout*: 234 M de horas en su lanzamiento (agosto de 2020) quedaban como "0 horas", porque Steam lo renombró *Fall Guys*.

**Criterio para aceptar un alias** (los 3 se deben cumplir):
1. Es **el mismo producto de PC** que en Steam, solo renombrado o con un sufijo de edición (*GOTY*, *Anniversary*, *2025 Edition*...).
2. Se revisó a mano: la clave destino existe en Steam y la fecha es coherente con el período de Twitch.
3. **No** se mapean ports de consola a PC con varios años de diferencia (*Ghost of Tsushima*, *Death Stranding*, *FF VII Remake*), porque las horas de Twitch corresponden al lanzamiento en consola. Tampoco se mapean categorías que agrupan una franquicia o que son ambiguas (*Call of Duty: Warzone* → "Call of Duty®", *Age of Empires II*, *Dark Souls*).

In [30]:
ALIAS_TWITCH_A_STEAM = {
    # alias originales
    "playerunknownsbattlegrounds": "pubgbattlegrounds",
    "counterstrikeglobaloffensive": "counterstrike2",
    "grandtheftautov": "grandtheftautovlegacy",
    "tomclancysrainbowsixsiege": "tomclancysrainbowsixsiegex",
    "fifa21": "easportsfifa21",
    # alias agregados tras revisar twitch_sin_match.csv
    "fallguysultimateknockout": "fallguys",
    "newworld": "newworldaeternum",
    "seaofthieves": "seaofthieves2025edition",
    "blackdesertonline": "blackdesert",
    "huntshowdown": "huntshowdown1896",
    "sekiroshadowsdietwice": "sekiroshadowsdietwicegotyedition",
    "ageofempiresiv": "ageofempiresivanniversaryedition",
    "eternalreturnblacksurvival": "eternalreturn",
    "dyinglight2stayhuman": "dyinglight2stayhumanreloadededition",
    "fifa23": "easportsfifa23",
    "counterstrike2limitedtest": "counterstrike2",
    "microsoftflightsimulator": "microsoftflightsimulator202040thanniversaryedition",
    "microsoftflightsimulatorx": "microsoftflightsimulatorxsteamedition",
    "shadowverse": "shadowverseccg",
    "masseffectandromeda": "masseffectandromedadeluxeedition",
    "callofdutymodernwarfareremastered": "callofdutymodernwarfareremastered2017",
}

# Verificación: cada alias debe apuntar a una clave que exista en Steam limpio
destinos_faltantes = set(ALIAS_TWITCH_A_STEAM.values()) - set(steam_limpio["clave"])
assert not destinos_faltantes, f"Alias que apuntan a claves inexistentes en Steam: {destinos_faltantes}"

twitch["clave"] = twitch["clave_original"].replace(ALIAS_TWITCH_A_STEAM)
con_alias = twitch["clave"] != twitch["clave_original"]
log(f"Alias definidos: {len(ALIAS_TWITCH_A_STEAM)}; filas Twitch afectadas: {con_alias.sum():,}")

tabla_alias = (twitch[con_alias].groupby(["juego", "clave"])
               .agg(meses=("fecha", "size"), horas=("Hours_watched", "sum"))
               .merge(steam_limpio.set_index("clave")[["nombre", "fecha_lanzamiento"]],
                      left_on="clave", right_index=True)
               .rename(columns={"nombre": "nombre_steam"})
               .sort_values("horas", ascending=False))
tabla_alias

Alias definidos: 21; filas Twitch afectadas: 868


,,meses,horas,nombre_steam,fecha_lanzamiento
juego,clave,,,,
Grand Theft Auto V,grandtheftautovlegacy,105,7482026514,Grand Theft Auto V Legacy,2015-04-13
Counter-Strike: Global Offensive,counterstrike2,89,4315553263,Counter-Strike 2,2012-08-21
PLAYERUNKNOWN'S BATTLEGROUNDS,pubgbattlegrounds,55,1475296705,PUBG: BATTLEGROUNDS,2017-12-21
Tom Clancy's Rainbow Six Siege,tomclancysrainbowsixsiegex,105,1077985583,Tom Clancy's Rainbow Six® Siege X,2015-12-01
FIFA 21,easportsfifa21,13,403466710,EA SPORTS™ FIFA 21,2020-10-08
Sea of Thieves,seaofthieves2025edition,81,373694955,Sea of Thieves: 2025 Edition,2020-06-03
FIFA 23,easportsfifa23,13,326537693,EA SPORTS™ FIFA 23,2022-09-30
New World,newworldaeternum,29,277890359,New World: Aeternum,2021-09-28
Fall Guys: Ultimate Knockout,fallguys,18,234119887,Fall Guys,2020-08-03


### 4.6 Grano juego-mes: duplicados después de normalizar y aplicar alias

El grano de la tabla de hechos es **un juego por mes**. Al normalizar y aplicar alias, dos nombres de Twitch pueden quedar con la misma clave en el mismo mes (por ejemplo, el mes en que Twitch pasó de *Fall Guys: Ultimate Knockout* a *Fall Guys*).

In [31]:
dup_mes = twitch.duplicated(["clave", "fecha"], keep=False)
log(f"Filas juego-mes repetidas a consolidar: {twitch.duplicated(['clave', 'fecha']).sum()}")
twitch.loc[dup_mes, ["juego", "clave", "fecha", "Hours_watched", "Peak_viewers", "Streamers", "Rank"]].sort_values(["clave", "fecha"])

Filas juego-mes repetidas a consolidar: 12


,juego,clave,fecha,Hours_watched,Peak_viewers,Streamers,Rank
16867,Among Us,amongus,2023-01-01,2760266,137287,37885,68
16955,Among Us,amongus,2023-01-01,876724,135976,8939,156
7214,Atlas,atlas,2019-01-01,12110031,69614,10070,15
7249,Atlas,atlas,2019-01-01,2106560,75774,3909,50
17204,Counter-Strike: Global Offensive,counterstrike2,2023-03-01,65844705,889692,156919,5
17245,Counter-Strike 2,counterstrike2,2023-03-01,6127365,138445,3958,46
17404,Counter-Strike: Global Offensive,counterstrike2,2023-04-01,81024311,579417,173372,5
17469,Counter-Strike 2,counterstrike2,2023-04-01,2921735,69337,3165,70
18413,Counter-Strike 2 Limited Test,counterstrike2,2023-09-01,24443732,208933,62991,14
18540,Counter-Strike 2,counterstrike2,2023-09-01,924571,72929,12374,141


**Decisión de agregación** (depende de si la métrica es aditiva o no):

| Métrica | Agregación | Por qué |
|---|---|---|
| `Hours_watched`, `Hours_streamed` | **suma** | Son horas acumuladas en el mes: dos categorías del mismo juego suman horas. |
| `Peak_viewers`, `Peak_channels` | **máximo** | Un pico no se suma: el pico conjunto no puede superar el mayor de los dos. |
| `Streamers`, `Avg_viewers`, `Avg_channels` | **máximo** | Son conteos o promedios del mes. Sumarlos contaría dos veces a las mismas personas. |
| `Rank` | **mínimo** | La mejor posición alcanzada en ese mes. |

`Streamers` es un **conteo mensual**, no un identificador de personas, así que no permite calcular streamers únicos a lo largo de varios meses (ver P4).

In [32]:
AGREGACIONES = {
    "juego": "first", "Hours_watched": "sum", "Hours_streamed": "sum",
    "Peak_viewers": "max", "Peak_channels": "max", "Streamers": "max",
    "Avg_viewers": "max", "Avg_channels": "max", "Rank": "min",
}
antes = len(twitch)
twitch_limpio = (twitch.groupby(["clave", "fecha"], as_index=False).agg(AGREGACIONES)
                       .rename(columns=str.lower))
registrar_paso("Twitch", "Consolidar duplicados juego-mes", antes, len(twitch_limpio))

assert not twitch_limpio.duplicated(["clave", "fecha"]).any()
assert twitch_limpio["hours_watched"].sum() == twitch["Hours_watched"].sum(), "La agregación perdió horas"
log(f"Twitch limpio: {len(twitch_raw):,} -> {len(twitch_limpio):,} filas juego-mes")
twitch_limpio.head()

[Twitch] Consolidar duplicados juego-mes: 19,109 -> 19,097 (eliminadas: 12)


Twitch limpio: 21,000 -> 19,097 filas juego-mes


,clave,fecha,juego,hours_watched,hours_streamed,peak_viewers,peak_channels,streamers,avg_viewers,avg_channels,rank
0,20minutestilldawn,2022-06-01,20 Minutes Till Dawn,911356,12253,29743,80,3616,1267,17,159
1,2xko,2024-08-01,2XKO,2137305,31942,60961,508,6619,2876,42,94
2,60parsecs,2018-09-01,60 Parsecs!,529688,1867,31960,27,606,736,2,109
3,60seconds,2016-07-01,60 Seconds!,268754,597,32505,10,275,361,0,126
4,60seconds,2016-08-01,60 Seconds!,772786,2065,56904,30,628,1040,2,54


## 5. Merge entre fuentes

**Clave de cruce:** `clave`, el nombre normalizado más los alias de 4.5. Es una **clave natural construida**, porque las fuentes no comparten un ID. En Steam es única (paso 3.9). En Twitch se repite una vez por mes.

Se hacen **dos merges**, porque cada pregunta necesita un universo distinto:

| Merge | Tipo | Universo que se conserva | Por qué este join |
|---|---|---|---|
| A. `twitch_limpio` ⟕ `steam_limpio` | LEFT, `m:1` | todas las filas juego-mes de Twitch | Permite **medir la cobertura** (qué parte de Twitch encuentra su juego en Steam) y exportar los no cruzados para diagnóstico. Un INNER JOIN los ocultaría. |
| B. `steam_limpio` ⟕ métricas Twitch por juego | LEFT, `1:1` | todos los juegos de Steam | P1, P3 y el KPI 3 necesitan también los juegos **que nunca aparecieron** en el top 200. Un INNER JOIN dejaría solo los exitosos (sesgo de selección) y el KPI 3 daría siempre 100 %. |

`validate="m:1"` y `validate="1:1"` hacen que pandas **corte el proceso** si la clave no tiene la cardinalidad esperada, en lugar de multiplicar filas en silencio.

### 5.1 Merge A: hecho mensual (Twitch LEFT JOIN Steam)

In [33]:
log("\n" + "=" * 70 + "\n5. MERGE\n" + "=" * 70)
COLS_REF_STEAM = ["clave", "AppID", "nombre", "fecha_lanzamiento", "en_ventana",
                  "owners_medio", "genero_principal"]

mensual = twitch_limpio.merge(
    steam_limpio[COLS_REF_STEAM], on="clave", how="left", validate="m:1", indicator=True,
)
mensual["cruzo"] = mensual["_merge"].eq("both")
mensual = mensual.drop(columns="_merge")
mensual["AppID"] = mensual["AppID"].astype("Int64")   # sin esto queda float (730.0) por los nulos del LEFT JOIN
log(f"Merge A: {len(twitch_limpio):,} filas Twitch -> {len(mensual):,} filas (cruzan: {mensual['cruzo'].sum():,})")
mensual["cruzo"].value_counts().rename({True: "cruza con Steam", False: "sin cruce"}).to_frame("filas")


5. MERGE


Merge A: 19,097 filas Twitch -> 19,097 filas (cruzan: 11,260)


,filas
cruzo,
cruza con Steam,11260
sin cruce,7837


### 5.2 Mes relativo al lanzamiento (KPI 1)

`mes_rel = (año_twitch − año_lanzamiento)·12 + (mes_twitch − mes_lanzamiento)`. El **mes 0** es el mes calendario del lanzamiento, y 1 y 2 son los dos meses siguientes. Un valor negativo indica que el juego ya aparecía en Twitch antes de su fecha en Steam (betas, acceso anticipado en otra tienda o un cruce incorrecto).

In [34]:
f_tw, f_st = mensual["fecha"], mensual["fecha_lanzamiento"]
mensual["mes_rel"] = (f_tw.dt.year - f_st.dt.year) * 12 + (f_tw.dt.month - f_st.dt.month)
mensual["mes_rel"] = mensual["mes_rel"].astype("Int64")   # entero que admite nulos (filas sin cruce)
mensual.loc[mensual["cruzo"], "mes_rel"].describe()[["min", "25%", "50%", "75%", "max"]].to_frame("mes_rel").T

,min,25%,50%,75%,max
mes_rel,-119.0,0.0,16.0,50.0,286.0


### 5.3 Métricas por juego

Se agregan dos grupos de métricas para cada juego que cruzó:
- **Historia completa en Twitch:** horas totales, cantidad de meses en el top, primer y último mes, y pico máximo.
- **Antes del lanzamiento (mes_rel < 0):** `horas_pre_lanzamiento`, que se usa en el paso 5.5 para detectar fechas de Steam que no son el lanzamiento real.
- **Ventana de lanzamiento (meses 0, 1 y 2):** `horas_3m` (KPI 1), `streamers_max_3m` y `pico_espectadores_3m` (P4).

In [35]:
cruzados = mensual[mensual["cruzo"]]
por_juego = cruzados.groupby("clave").agg(
    horas_totales=("hours_watched", "sum"),
    meses_en_top=("fecha", "nunique"),
    primer_mes_twitch=("fecha", "min"),
    ultimo_mes_twitch=("fecha", "max"),
    primer_mes_rel=("mes_rel", "min"),
    pico_espectadores_max=("peak_viewers", "max"),
    streamers_max_mensual=("streamers", "max"),
)
por_juego["horas_pre_lanzamiento"] = (cruzados[cruzados["mes_rel"] < 0]
                                      .groupby("clave")["hours_watched"].sum()
                                      .reindex(por_juego.index, fill_value=0))
lanzamiento = cruzados[cruzados["mes_rel"].between(0, 2)].groupby("clave").agg(
    horas_3m=("hours_watched", "sum"),
    meses_top_3m=("fecha", "nunique"),
    streamers_max_3m=("streamers", "max"),
    pico_espectadores_3m=("peak_viewers", "max"),
)
metricas_juego = por_juego.join(lanzamiento, how="left").reset_index()
print(f"Juegos de Steam con al menos un mes en el top 200: {len(metricas_juego):,}")
metricas_juego.sort_values("horas_3m", ascending=False).head()

Juegos de Steam con al menos un mes en el top 200: 1,361


,clave,horas_totales,meses_en_top,primer_mes_twitch,ultimo_mes_twitch,primer_mes_rel,pico_espectadores_max,streamers_max_mensual,horas_pre_lanzamiento,horas_3m,meses_top_3m,streamers_max_3m,pico_espectadores_3m
357,eldenring,590668921,33,2021-11-01,2024-09-01,-3,908307,434645,3040515,230162782.0,3.0,434645.0,908307.0
663,lostark,662177548,68,2017-09-01,2024-09-01,-53,1273854,196480,170814671,212333622.0,3.0,196480.0,1273854.0
388,fallguys,416136517,51,2020-07-01,2024-09-01,-1,713229,504731,4961218,160941374.0,3.0,395663.0,713229.0
875,pubgbattlegrounds,1865548227,95,2016-09-01,2024-09-01,-15,579861,252217,486132714,152541322.0,3.0,252217.0,482947.0
354,easportsfifa21,403466710,13,2020-09-01,2021-09-01,-1,837094,137198,4493195,151604082.0,3.0,104490.0,474733.0


### 5.4 Merge B: universo de juegos (Steam LEFT JOIN métricas)

Este LEFT JOIN **genera nulos a propósito**: todos los juegos que nunca entraron al top 200 quedan sin métricas de Twitch. Primero **se cuentan** y después se decide qué hacer con cada columna:

In [36]:
juegos = steam_limpio.merge(metricas_juego, on="clave", how="left", validate="1:1")
log(f"Merge B: {len(steam_limpio):,} juegos Steam -> {len(juegos):,} filas")

COLS_METRICAS = ["horas_totales", "meses_en_top", "pico_espectadores_max", "streamers_max_mensual",
                 "horas_pre_lanzamiento", "horas_3m", "meses_top_3m", "streamers_max_3m", "pico_espectadores_3m"]
COLS_FECHA_TW = ["primer_mes_twitch", "ultimo_mes_twitch", "primer_mes_rel"]
nulos_merge_b = juegos[COLS_METRICAS + COLS_FECHA_TW].isna().sum().to_frame("nulos_introducidos")
nulos_merge_b

Merge B: 99,596 juegos Steam -> 99,596 filas


,nulos_introducidos
horas_totales,98235
meses_en_top,98235
pico_espectadores_max,98235
streamers_max_mensual,98235
horas_pre_lanzamiento,98235
horas_3m,98693
meses_top_3m,98693
streamers_max_3m,98693
pico_espectadores_3m,98693
primer_mes_twitch,98235


**Decisión sobre esos nulos:**
- **Métricas** (horas, picos, streamers, meses) → **0**. Que un juego no esté en el top 200 significa que su audiencia quedó **por debajo del umbral observable**, no que nadie lo transmitió. Por eso el 0 se interpreta como "visibilidad bajo el umbral" y P1 informa esos juegos **aparte**: los cuartiles se calculan solo con `horas_3m > 0`.
- **Fechas** (`primer_mes_twitch`, etc.) → **quedan nulas**. No existe un "primer mes" para un juego que nunca apareció, y poner una fecha inventada sería un error.

Se crean dos marcas para no mezclar situaciones distintas que comparten `horas_3m = 0`:
- `en_twitch`: apareció **alguna vez** en el top 200 (es lo que usan P3 y el KPI 3).
- `en_top_lanzamiento`: apareció **durante sus 3 primeros meses** (P1). Un juego como *Among Us* tiene `en_twitch = True` pero `en_top_lanzamiento = False`, porque se hizo popular en Twitch dos años después de salir.

In [37]:
juegos[COLS_METRICAS] = juegos[COLS_METRICAS].fillna(0)
juegos["primer_mes_rel"] = juegos["primer_mes_rel"].astype("Int64")
juegos["en_twitch"] = juegos["meses_en_top"] > 0
juegos["en_top_lanzamiento"] = juegos["horas_3m"] > 0

pd.crosstab(juegos["en_twitch"], juegos["en_top_lanzamiento"], margins=True)

en_top_lanzamiento,False,True,All
en_twitch,,,
False,98235,0,98235
True,458,903,1361
All,98693,903,99596


### 5.5 Cruces sospechosos y cohorte de P1

**Problema:** P1 mide las horas de los meses 0 a 2 desde la fecha de Steam. Si esa fecha **no es el lanzamiento real** del juego (porque salió antes en otra tienda o en consola, o porque Steam guarda la fecha de salida del acceso anticipado), las "horas de lanzamiento" miden otra cosa y el juego tiene que quedar fuera de P1.

**Regla anterior:** era sospechoso cualquier juego que apareciera en Twitch **más de un mes antes** de su fecha de Steam (`primer_mes_rel < −1`). Se evaluó con los datos:

In [38]:
juegos["sospechoso_regla_anterior"] = juegos["en_twitch"] & (juegos["primer_mes_rel"] < -1).fillna(False)
juegos["match_sospechoso"] = juegos["en_twitch"] & (juegos["horas_pre_lanzamiento"] > juegos["horas_3m"])

comparacion = juegos.loc[juegos["en_ventana"] & (juegos["sospechoso_regla_anterior"] | juegos["match_sospechoso"]),
                         ["nombre", "fecha_lanzamiento", "primer_mes_rel", "horas_pre_lanzamiento", "horas_3m",
                          "sospechoso_regla_anterior", "match_sospechoso"]]
comparacion.sort_values("horas_3m", ascending=False).head(16)

,nombre,fecha_lanzamiento,primer_mes_rel,horas_pre_lanzamiento,horas_3m,sospechoso_regla_anterior,match_sospechoso
17,ELDEN RING,2022-02-24,-3,3040515.0,230162782.0,True,False
12,Lost Ark,2022-02-11,-53,170814671.0,212333622.0,True,False
1,PUBG: BATTLEGROUNDS,2017-12-21,-15,486132714.0,152541322.0,True,True
11,New World: Aeternum,2021-09-28,-13,51191114.0,141476225.0,True,False
3,Apex Legends™,2020-11-04,-21,588312662.0,115638437.0,True,True
21,Cyberpunk 2077,2020-12-09,-28,1717993.0,68273873.0,True,False
408,Street Fighter™ 6,2023-06-01,-8,7980042.0,65244023.0,True,False
24,Baldur's Gate 3,2023-08-03,-34,7149380.0,60339306.0,True,False
95797,Overwatch® 2,2023-08-10,-16,374920146.0,56981638.0,True,True
132,Resident Evil 4,2023-03-23,-84,8702069.0,53221504.0,True,False


**Evidencia:** la regla anterior excluía también los **lanzamientos más grandes y reales**, solo porque el juego había aparecido antes en Twitch por una beta, un *network test*, un tráiler o un acceso anticipado: *Elden Ring* (3 M de horas antes, 230 M en el lanzamiento), *Cyberpunk 2077*, *Baldur's Gate 3*, *Street Fighter 6*, *Resident Evil 4* (remake). Sacar justamente esos juegos sesga P1 hacia abajo en el cuartil más alto.

**Regla nueva (la que se usa):** un cruce es sospechoso cuando el juego **acumuló en Twitch más horas antes de su fecha de Steam que en sus 3 meses de "lanzamiento"** (`horas_pre_lanzamiento > horas_3m`). Si antes de la fecha ya se veía más que después, esa fecha no es su lanzamiento real.
- Siguen excluidos: *PUBG* (acceso anticipado masivo en 2017), *Apex Legends* (Origin, 2019), *Overwatch 2* (Battle.net), *Destiny 2*, *Diablo IV*, *Sea of Thieves*, *Monster Hunter: World* (consola).
- Vuelven a P1: *Elden Ring*, *Cyberpunk 2077*, *Baldur's Gate 3*, *Street Fighter 6*, *Halo Infinite*, *New World*.

`sospechoso_regla_anterior` queda en la salida para que se pueda comparar. **Para volver a la regla anterior** alcanza con usar esa columna en `apto_p1`.

In [39]:
juegos["apto_p1"] = juegos["en_ventana"] & ~juegos["match_sospechoso"]

en_v = juegos["en_ventana"]
log(f"Cruces sospechosos en la ventana - regla anterior: {(en_v & juegos['sospechoso_regla_anterior']).sum():,} | "
    f"regla nueva: {(en_v & juegos['match_sospechoso']).sum():,}")


def mediana_owners_por_cuartil(mascara_excluir):
    base = juegos[en_v & ~mascara_excluir & (juegos["horas_3m"] > 0)]
    cuartil = pd.qcut(base["horas_3m"], 4, labels=["Q1", "Q2", "Q3", "Q4"])
    return base.groupby(cuartil, observed=True)["owners_medio"].median()


pd.DataFrame({"regla anterior": mediana_owners_por_cuartil(juegos["sospechoso_regla_anterior"]),
              "regla nueva": mediana_owners_por_cuartil(juegos["match_sospechoso"])}).rename_axis("cuartil horas_3m")

Cruces sospechosos en la ventana - regla anterior: 352 | regla nueva: 273


,regla anterior,regla nueva
cuartil horas_3m,,
Q1,350000.0,350000.0
Q2,750000.0,750000.0
Q3,750000.0,750000.0
Q4,1500000.0,3500000.0


La tabla muestra la mediana de owners por cuartil de horas con cada regla. Con la regla anterior, el Q4 quedaba subestimado porque le faltaban los lanzamientos más vistos. Esta comparación también sirve como **análisis de sensibilidad** para el informe: el patrón general (más horas, más owners) se mantiene con las dos reglas.

## 6. Validaciones posteriores al merge

Cada validación usa `assert`: si falla, el notebook se detiene y no exporta datos inconsistentes.

### 6.1 Cantidad de filas antes y después

In [40]:
log("\n" + "=" * 70 + "\n6. VALIDACIONES DEL MERGE\n" + "=" * 70)
assert len(mensual) == len(twitch_limpio), "El merge A multiplicó o perdió filas"
assert len(juegos) == len(steam_limpio), "El merge B multiplicó o perdió filas"
log(f"Filas merge A: {len(twitch_limpio):,} antes = {len(mensual):,} después  OK")
log(f"Filas merge B: {len(steam_limpio):,} antes = {len(juegos):,} después  OK")

pd.DataFrame(TRAZA)


6. VALIDACIONES DEL MERGE
Filas merge A: 19,097 antes = 19,097 después  OK
Filas merge B: 99,596 antes = 99,596 después  OK


,fuente,paso,filas_antes,filas_despues,eliminadas
0,Steam,Eliminar filas sin nombre,125855,125854,1
1,Steam,Eliminar AppID duplicados,125854,125854,0
2,Steam,Excluir owners '0 - 0',125854,101276,24578
3,Steam,Excluir demos/playtests/soundtracks/servidores,101276,101227,49
4,Steam,Resolver nombres repetidos (1 AppID por clave),101227,99596,1631
5,Twitch,Eliminar filas sin nombre de juego,21000,20999,1
6,Twitch,Excluir categorías que no son juegos,20999,19109,1890
7,Twitch,Consolidar duplicados juego-mes,19109,19097,12


### 6.2 Nulos introducidos por el merge

En un LEFT JOIN, las columnas de la tabla **izquierda** nunca reciben nulos nuevos. Los nulos aparecen en las columnas de la tabla **derecha**, en las filas sin pareja. Por eso se validan los dos lados:
- Columnas de Twitch en el merge A: deben tener **0** nulos nuevos.
- Columnas de Steam en el merge A: la cantidad de nulos debe ser **exactamente igual** a la cantidad de filas sin cruce. Si fuera mayor, significaría que hay filas que cruzaron pero traen datos vacíos.
- Merge B: las columnas históricas deben tener tantos nulos como juegos que nunca aparecieron en el top 200. Las de la ventana de 3 meses tienen además los juegos que aparecieron, pero no durante su lanzamiento.

In [41]:
cols_tw = twitch_limpio.columns.tolist()
nuevos_tw = int((mensual[cols_tw].isna().sum() - twitch_limpio[cols_tw].isna().sum()).sum())
assert nuevos_tw == 0
log(f"Merge A - nulos nuevos en columnas Twitch: {nuevos_tw}  OK")

sin_cruce_n = int((~mensual["cruzo"]).sum())
cols_st = [c for c in COLS_REF_STEAM if c != "clave"]
tabla_nulos = mensual[cols_st].isna().sum().to_frame("nulos")
tabla_nulos["filas_sin_cruce"] = sin_cruce_n
tabla_nulos["coincide"] = tabla_nulos["nulos"] == sin_cruce_n
assert tabla_nulos["coincide"].all(), "Hay filas que cruzaron con datos de Steam vacíos"
log(f"Merge A - nulos en columnas Steam = filas sin cruce ({sin_cruce_n:,}) en todas las columnas  OK")
display(tabla_nulos)

log("Merge B - nulos introducidos antes de imputar:")
for col, n in nulos_merge_b["nulos_introducidos"].items():
    log(f"   {col}: {n:,}")
sin_twitch = len(juegos) - len(metricas_juego)
sin_lanzamiento = len(juegos) - int(metricas_juego["horas_3m"].notna().sum())
COLS_HIST = ["horas_totales", "meses_en_top", "pico_espectadores_max", "streamers_max_mensual",
             "horas_pre_lanzamiento"] + COLS_FECHA_TW
COLS_3M = ["horas_3m", "meses_top_3m", "streamers_max_3m", "pico_espectadores_3m"]
assert (nulos_merge_b.loc[COLS_HIST, "nulos_introducidos"] == sin_twitch).all()
assert (nulos_merge_b.loc[COLS_3M, "nulos_introducidos"] == sin_lanzamiento).all()
assert juegos[COLS_METRICAS].isna().sum().sum() == 0
log(f"Merge B - nulos históricos = juegos nunca vistos en el top 200 ({sin_twitch:,})  OK")
log(f"Merge B - nulos de ventana 3m = juegos sin top 200 en sus meses 0-2 ({sin_lanzamiento:,}; "
    f"incluye {sin_lanzamiento - sin_twitch:,} que sí aparecieron en otros meses)  OK")
log("Merge B - métricas imputadas a 0; fechas Twitch se mantienen nulas  OK")

Merge A - nulos nuevos en columnas Twitch: 0  OK
Merge A - nulos en columnas Steam = filas sin cruce (7,837) en todas las columnas  OK


,nulos,filas_sin_cruce,coincide
AppID,7837,7837,True
nombre,7837,7837,True
fecha_lanzamiento,7837,7837,True
en_ventana,7837,7837,True
owners_medio,7837,7837,True
genero_principal,7837,7837,True


Merge B - nulos introducidos antes de imputar:
   horas_totales: 98,235
   meses_en_top: 98,235
   pico_espectadores_max: 98,235
   streamers_max_mensual: 98,235
   horas_pre_lanzamiento: 98,235
   horas_3m: 98,693
   meses_top_3m: 98,693
   streamers_max_3m: 98,693
   pico_espectadores_3m: 98,693
   primer_mes_twitch: 98,235
   ultimo_mes_twitch: 98,235
   primer_mes_rel: 98,235
Merge B - nulos históricos = juegos nunca vistos en el top 200 (98,235)  OK
Merge B - nulos de ventana 3m = juegos sin top 200 en sus meses 0-2 (98,693; incluye 458 que sí aparecieron en otros meses)  OK
Merge B - métricas imputadas a 0; fechas Twitch se mantienen nulas  OK


### 6.3 Duplicados

In [42]:
dup_juego = int(juegos["clave"].duplicated().sum())
dup_appid = int(juegos["AppID"].duplicated().sum())
dup_mes = int(mensual.duplicated(["clave", "fecha"]).sum())
assert dup_juego == dup_appid == dup_mes == 0
log(f"Duplicados -> clave en juegos: {dup_juego}, AppID en juegos: {dup_appid}, juego-mes en mensual: {dup_mes}  OK")

Duplicados -> clave en juegos: 0, AppID en juegos: 0, juego-mes en mensual: 0  OK


### 6.4 Integridad del cruce

- **Conservación de horas:** horas cruzadas + horas sin cruce = horas originales (sin categorías que no son juegos). Así se comprueba que el merge no perdió ni duplicó horas.
- **Integridad referencial:** cada fila mensual que cruzó tiene `AppID`, y ese `AppID` existe en la tabla de juegos. Es la condición que exige la clave foránea de la tabla de hechos en el DW.
- **Consistencia entre tablas:** las horas totales de `juegos` coinciden con las horas cruzadas de `mensual`.

In [43]:
h_orig = twitch_limpio["hours_watched"].sum()
h_cruz = mensual.loc[mensual["cruzo"], "hours_watched"].sum()
h_sin = mensual.loc[~mensual["cruzo"], "hours_watched"].sum()
assert h_cruz + h_sin == h_orig
log(f"Horas: {h_cruz:,} cruzadas + {h_sin:,} sin cruce = {h_orig:,} originales  OK")

assert mensual.loc[mensual["cruzo"], "AppID"].notna().all()
assert mensual.loc[mensual["cruzo"], "AppID"].isin(juegos["AppID"]).all()
log("Integridad referencial mensual.AppID -> juegos.AppID  OK")

assert juegos["horas_totales"].sum() == h_cruz
log("Horas totales en juegos = horas cruzadas en mensual  OK")

assert not juegos.loc[~juegos["clave_cruzable"], "en_twitch"].any()
log("Juegos con clave no cruzable sin cruces accidentales  OK")

Horas: 50,703,033,431 cruzadas + 47,485,055,507 sin cruce = 98,188,088,938 originales  OK
Integridad referencial mensual.AppID -> juegos.AppID  OK
Horas totales en juegos = horas cruzadas en mensual  OK
Juegos con clave no cruzable sin cruces accidentales  OK


### 6.5 Cobertura del cruce

In [44]:
cobertura = mensual.groupby("clave")["cruzo"].first()
log(f"Cobertura de juegos Twitch: {cobertura.sum():,}/{len(cobertura):,} ({cobertura.mean():.1%})")
log(f"Cobertura de horas vistas: {h_cruz / h_orig:.1%}")

sin_cruce = (mensual[~mensual["cruzo"]].groupby("clave")
             .agg(juego=("juego", "first"), horas=("hours_watched", "sum"), meses=("fecha", "nunique"))
             .sort_values("horas", ascending=False).reset_index())
sin_cruce.head(12)

Cobertura de juegos Twitch: 1,361/2,278 (59.7%)
Cobertura de horas vistas: 51.6%


,clave,juego,horas,meses
0,leagueoflegends,League of Legends,11296781271,105
1,fortnite,Fortnite,6224642826,87
2,valorant,VALORANT,4631862775,54
3,minecraft,Minecraft,3162054933,105
4,worldofwarcraft,World of Warcraft,2870508014,105
5,callofdutywarzone,Call of Duty: Warzone,2764017645,62
6,hearthstone,Hearthstone,2513198715,105
7,overwatch,Overwatch,1393597856,80
8,teamfighttactics,Teamfight Tactics,1169318581,64
9,genshinimpact,Genshin Impact,547266089,50


**Lectura de la cobertura:** la mayor parte de las horas sin cruce corresponde a juegos que **no están en Steam** (*League of Legends*, *Fortnite*, *Valorant*, *Minecraft*, *World of Warcraft*, *Hearthstone*, *Overwatch*) o a juegos de consola o móvil. No es un error del ETL: es una limitación de las fuentes y se informa en el alcance del trabajo. La lista completa se exporta en `twitch_sin_match.csv` para auditarla.

### 6.6 Control de casos conocidos

Se revisan a mano juegos cuyo comportamiento en Twitch es conocido, para confirmar que los resultados tienen sentido:

In [45]:
CASOS = ["Fall Guys", "New World: Aeternum", "ELDEN RING", "Palworld", "HELLDIVERS™ 2", "Among Us",
         "Counter-Strike 2", "Apex Legends™", "Sea of Thieves: 2025 Edition", "Euro Truck Simulator 2"]
control = juegos.loc[juegos["nombre"].isin(CASOS),
                     ["nombre", "fecha_lanzamiento", "genero_principal", "owners_medio", "horas_3m",
                      "en_twitch", "en_top_lanzamiento", "match_sospechoso", "apto_p1"]]
control

,nombre,fecha_lanzamiento,genero_principal,owners_medio,horas_3m,en_twitch,en_top_lanzamiento,match_sospechoso,apto_p1
0,Counter-Strike 2,2012-08-21,Action,150000000.0,0.0,True,False,False,False
3,Apex Legends™,2020-11-04,Action,150000000.0,115638437.0,True,True,True,False
10,Palworld,2024-01-18,Adventure,75000000.0,77749506.0,True,True,False,True
11,New World: Aeternum,2021-09-28,Massively Multiplayer,75000000.0,141476225.0,True,True,False,True
17,ELDEN RING,2022-02-24,RPG,35000000.0,230162782.0,True,True,False,True
18,HELLDIVERS™ 2,2024-02-08,Action,35000000.0,41446033.0,True,True,False,True
27,Among Us,2018-11-16,Casual,35000000.0,0.0,True,False,False,True
32,Fall Guys,2020-08-03,Massively Multiplayer,35000000.0,160941374.0,True,True,False,True
44,Euro Truck Simulator 2,2012-10-12,Simulation,15000000.0,0.0,True,False,False,False
99,Sea of Thieves: 2025 Edition,2020-06-03,Adventure,7500000.0,23250101.0,True,True,True,False


In [46]:
def caso(clave):
    return juegos.loc[juegos["clave"] == clave].iloc[0]

assert caso("fallguys")["horas_3m"] > 0 and caso("fallguys")["apto_p1"], "Fall Guys: debería ser apto con horas"
assert caso("eldenring")["apto_p1"], "Elden Ring: lanzamiento real, debería ser apto"
assert not caso("apexlegends")["apto_p1"], "Apex: salió antes en Origin, no debería ser apto"
assert caso("eurotrucksimulator2")["genero_principal"] == "Simulation", "ETS2 debería ser Simulation"
log("Controles de casos conocidos (Fall Guys, Elden Ring, Apex, Euro Truck Simulator 2)  OK")

Controles de casos conocidos (Fall Guys, Elden Ring, Apex, Euro Truck Simulator 2)  OK

Los resultados son los esperados:
- *Fall Guys*, *Elden Ring* y *New World* tienen sus horas de lanzamiento y son aptos para P1.
- *Apex Legends* y *Sea of Thieves* quedan excluidos de P1 porque salieron antes en otra tienda, pero siguen contando para P3.
- *Among Us* aparece en Twitch, aunque no durante su lanzamiento.
- *Euro Truck Simulator 2* queda clasificado como *Simulation*.

### 6.7 Tamaño de los grupos para cada pregunta

In [47]:
cohorte = juegos[juegos["apto_p1"]]
p3 = juegos[juegos["owners_medio"] >= 100_000]
resumen = pd.Series({
    "P1 - cohorte apta": len(cohorte),
    "P1 - sin aparición en el top 200 (informados aparte)": int((~cohorte["en_top_lanzamiento"]).sum()),
    "P1 - con horas_3m > 0 (base de los cuartiles)": int(cohorte["en_top_lanzamiento"].sum()),
    "P3 - juegos con >= 100.000 owners": len(p3),
    "P3 - de esos, presentes alguna vez en Twitch": int(p3["en_twitch"].sum()),
}).to_frame("juegos")
for k, v in resumen["juegos"].items():
    log(f"{k}: {v:,}")
resumen

P1 - cohorte apta: 70,435
P1 - sin aparición en el top 200 (informados aparte): 69,661
P1 - con horas_3m > 0 (base de los cuartiles): 774
P3 - juegos con >= 100.000 owners: 8,787
P3 - de esos, presentes alguna vez en Twitch: 1,238


,juegos
P1 - cohorte apta,70435
P1 - sin aparición en el top 200 (informados aparte),69661
P1 - con horas_3m > 0 (base de los cuartiles),774
P3 - juegos con >= 100.000 owners,8787
"P3 - de esos, presentes alguna vez en Twitch",1238


## 7. Exportación

| Archivo | Contenido | Uso en el DW |
|---|---|---|
| `steam_limpio.csv` | catálogo limpio, 1 fila por juego | base de la dimensión Juego |
| `twitch_limpio.csv` | Twitch juego-mes antes del merge | trazabilidad |
| `twitch_mensual.csv` | hecho mensual **solo de filas cruzadas** (tienen `AppID`) | tabla de hechos + DimTiempo |
| `juegos_merge.csv` | universo Steam con métricas Twitch y marcas de cohorte | dimensión Juego / consultas P1–P4 |
| `twitch_sin_match.csv` | diagnóstico de juegos de Twitch sin cruce | auditoría |
| `trazabilidad_filas.csv` | filas antes y después de cada paso | informe |
| `validaciones.txt` | reporte completo de esta corrida | evidencia de validación |

In [48]:
steam_limpio.to_csv(SALIDA / "steam_limpio.csv", index=False, encoding="utf-8-sig")
twitch_limpio.to_csv(SALIDA / "twitch_limpio.csv", index=False, encoding="utf-8-sig")
mensual[mensual["cruzo"]].drop(columns="cruzo").to_csv(SALIDA / "twitch_mensual.csv", index=False, encoding="utf-8-sig")
juegos.to_csv(SALIDA / "juegos_merge.csv", index=False, encoding="utf-8-sig")
sin_cruce.to_csv(SALIDA / "twitch_sin_match.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(TRAZA).to_csv(SALIDA / "trazabilidad_filas.csv", index=False, encoding="utf-8-sig")
(SALIDA / "validaciones.txt").write_text("\n".join(REPORTE), encoding="utf-8")

for archivo in sorted(SALIDA.iterdir()):
    print(f"{archivo.name:28} {archivo.stat().st_size / 1e6:8.2f} MB")

juegos_merge.csv                24.92 MB
steam_limpio.csv                18.06 MB
trazabilidad_filas.csv           0.00 MB
twitch_limpio.csv                1.62 MB
twitch_mensual.csv               1.65 MB
twitch_sin_match.csv             0.04 MB
validaciones.txt                 0.00 MB
